# Speculatores 14.5 Colab Runner

This notebook is a thin front end for the script-backed `Speculatores 14.5` pipeline.
Edit the config cell, run top to bottom, and the optimizer will write one timestamped Markdown report per run.

In [ ]:
# Cell 1 ? Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cell 2 ? Clone or update repo and install deps
import os
from pathlib import Path

REPO_DIR = Path('/content/cfd9')
REPO_URL = 'https://github.com/Sovenski/cfd9.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}
!pip install -q -r requirements.txt


In [ ]:
# Cell 3 ? Run config
from pathlib import Path

DATASET = 'data/raw/SPX_1D_18710201_20260318.csv'
TRIALS_PER_SIDE = 100
WORKERS_PER_SIDE = 2
STARTUP_TRIALS = 80
SEED = 42
SKIP_CROSS_ASSET = False

DRIVE_ROOT = Path('/content/drive/MyDrive/cfd9')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
STORAGE = DRIVE_ROOT / 'spec145_spx.journal'
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print({
    'dataset': DATASET,
    'trials_per_side': TRIALS_PER_SIDE,
    'workers_per_side': WORKERS_PER_SIDE,
    'startup_trials': STARTUP_TRIALS,
    'storage': str(STORAGE),
    'results_dir': str(RESULTS_DIR),
    'skip_cross_asset': SKIP_CROSS_ASSET,
})


In [ ]:
# Cell 4 ? Run Speculatores 14.5
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/cfd9').resolve()
assert REPO_DIR.exists(), REPO_DIR
assert (REPO_DIR / 'scripts' / 'run_speculatores_145.py').exists()

cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'run_speculatores_145.py'),
    '--dataset', str(REPO_DIR / DATASET),
    '--trials-per-side', str(TRIALS_PER_SIDE),
    '--workers-per-side', str(WORKERS_PER_SIDE),
    '--startup-trials', str(STARTUP_TRIALS),
    '--seed', str(SEED),
    '--storage', str(STORAGE),
    '--results-dir', str(RESULTS_DIR),
]
if SKIP_CROSS_ASSET:
    cmd.append('--skip-cross-asset')

print('RUNNING:')
print(' '.join(cmd))

res = subprocess.run(
    cmd,
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)

print('\nSTDOUT:\n')
print(res.stdout)
print('\nSTDERR:\n')
print(res.stderr)

if res.returncode != 0:
    raise RuntimeError(f'Speculatores 14.5 failed with exit code {res.returncode}')


In [ ]:
# Cell 5 ? Show latest report path and preview
from pathlib import Path
reports = sorted(Path(RESULTS_DIR).glob('*.md'), key=lambda p: p.stat().st_mtime, reverse=True)
assert reports, 'No reports found.'
latest_report = reports[0]
print(f'Latest report: {latest_report}')
print('
'.join(latest_report.read_text(encoding='utf-8').splitlines()[:120]))


In [ ]:
# Cell 6 ? Optional: inspect parity section only
text = latest_report.read_text(encoding='utf-8')
marker = '## Cell 3.3'
idx = text.find(marker)
if idx >= 0:
    print(text[idx:idx+2500])
else:
    print('Parity section not found.')
